> This notebook requires private benchmark data files that are not included in the public repository. See `../data/raw/README.md` and `../data/raw/labels/README.md`.

# Healthy Baseline EDA - Vibration RMS Features

This notebook explores the 29-scenario vibration benchmark along two lines:

1. **Self-historical baseline.** Each scenario's `fit` split defines its own healthy operating envelope. No peer context or fleet comparison is used — each sensor is evaluated against itself.

2. **Grouping surrogate.** The dataset is anonymous and carries no machine metadata. Observed failure archetypes from the `pred` split are used as a stand-in for asset families. In production, this grouping would come from known attributes such as machine type or operating context.

**Sections:** dataset → preprocessing contract → healthy baseline and residual calibration → degradation inspection → manual grouping.

> Model fitting, alert validation, and failure analysis are in `1.02-acp-model-debugging.ipynb`.

In [1]:
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

In [2]:
from offline_analysis.plotting import (
    create_group_distribution_widget,
    create_rms_scenario_widget,
    create_scenario_inspector,
    create_sigmoid_global_residual_widget,
    set_plot_style,
)
from anomaly_detection.model.grouped_residual.preprocessing import apply_norm_scores, fit_norm_baselines
from anomaly_detection.model.grouped_residual.preprocessing import clip_rms_spikes
from anomaly_detection.model.shared.scenario_groups import add_scenario_group_labels

set_plot_style()

# 1. Dataset

The dataset contains vibration measurements from **29 independent sensor scenarios**. Each scenario provides two time windows:

| Split | File pattern | Label |
|-------|-------------|-------|
| `fit` | `sensor_data_fit_{1..29}.parquet` | Always `"normal"` - used as the healthy reference |
| `pred` | `sensor_data_pred_{1..29}.parquet` | `"normal"` or `"incident_N"` - evaluated against the model |

**Key columns:**
- `sampled_at` - UTC timestamp of each measurement
- `uptime` - `True` when the machine is running; `False` during idle/off periods
- `vel_rms_{x,y,z}` - velocity RMS (mm/s) on three axes
- `accel_rms_{x,y,z}` - acceleration RMS (g) on three axes

Incident windows are defined in `the incident-label YAML` and applied to `pred` rows only. Each `pred` row is labelled `"normal"` or `"incident_N"` depending on whether its timestamp falls within a known failure window.

In [3]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "data").exists():
            return path
    return start.parent if start.name == "notebooks" else start


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
LABELS_PATH = PROJECT_ROOT / "data" / "raw" / "labels" / "incidents.yaml"

In [4]:
# --- load incident windows ---------------------------------------------------
with open(LABELS_PATH) as f:
    _raw = yaml.safe_load(f)

# {scenario_id: [(start, end, incident_index), ...]}
_incidents: dict[int, list[tuple]] = {
    int(sid): [
        (pd.Timestamp(w["start"]), pd.Timestamp(w["end"]), i)
        for i, w in enumerate(windows, start=1)
    ]
    for sid, windows in _raw.items()
}


def _assign_label(ts: pd.Timestamp, windows: list[tuple]) -> str:
    """Return 'incident_N' if ts falls in the Nth window, else 'normal'."""
    for start, end, idx in windows:
        if start <= ts <= end:
            return f"incident_{idx}"
    return "normal"


# --- load fit files (always healthy) ----------------------------------------
fit_frames = []
for path in sorted(DATA_DIR.glob("sensor_data_fit_*.parquet"), key=lambda p: int(p.stem.split("_")[-1])):
    scenario_id = int(path.stem.split("_")[-1])
    df = pd.read_parquet(path)
    df.insert(0, "scenario_id", scenario_id)
    df.insert(1, "split", "fit")
    df.insert(2, "label", "normal")
    fit_frames.append(df)

# --- load pred files (label rows against incident windows) ------------------
pred_frames = []
for path in sorted(DATA_DIR.glob("sensor_data_pred_*.parquet"), key=lambda p: int(p.stem.split("_")[-1])):
    scenario_id = int(path.stem.split("_")[-1])
    df = pd.read_parquet(path)
    df.insert(0, "scenario_id", scenario_id)
    df.insert(1, "split", "pred")
    windows = _incidents.get(scenario_id, [])
    df.insert(2, "label", df["sampled_at"].apply(lambda ts: _assign_label(ts, windows)))
    pred_frames.append(df)

# --- combine -----------------------------------------------------------------
full_df = (
    pd.concat(fit_frames + pred_frames, ignore_index=True)
    .sort_values(["scenario_id", "sampled_at"])
    .reset_index(drop=True)
)

In [5]:
summary = (
    full_df[['scenario_id', 'uptime', 'label']]
    .value_counts()
    .rename('count')
    .reset_index()
    .sort_values(['scenario_id', 'uptime', 'label'])
)


failure_labels = {'incident_1', 'incident_2'}
summary['failure'] = summary['label'].isin(failure_labels)

totals = (
    summary.groupby(['uptime', 'failure'])['count']
    .sum()
    .unstack(fill_value=0)
)

totals.columns = ['normal_count', 'failure_count']

pct = (
    totals.div(totals.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

pct.columns = ['normal_%', 'failure_%']

result = pd.concat([totals, pct], axis=1)
result

,normal_count,failure_count,normal_%,failure_%
uptime,,,,
False,41715,640,98.49,1.51
True,98113,3206,96.84,3.16


The distribution above reveals two important properties:

- **Incidents are rare** (< 4% of all rows) — severe class imbalance that rules out supervised classification. Unsupervised baseline deviation is the chosen approach.
- **Incidents occur only during `uptime=True`.** Anomalies are only measurable when the machine is running. Rows where `uptime=False` carry no incident signal and are excluded from baseline fitting and scoring.

## Anomaly Detection Framework Context

With 29 isolated sensors and no peer or fleet context, this is a **self-historical** problem: learn a healthy baseline from `fit`, look for deviations in `pred`. The one gating rule (full contract in Section 2) is trusting `uptime=True` alone — low amplitude, level shifts, or idle-looking behavior inside an uptime period are never reinterpreted as the machine being off.

A machine can stay "active" even as its RMS level shifts from load or operating condition, as long as `uptime=True` holds.

## Spike Clipping - Justification

`clip_rms_spikes(vel_threshold=100, accel_threshold=10)` removes sparse gross outliers before baseline fitting. These thresholds are orders of magnitude above the healthy operating band (velocity RMS is typically a few mm/s, acceleration RMS typically sub-g), so anything above them is a transient artifact.

Alternatives tested and rejected:

- **Butterworth / low-pass filtering.** Smears fast-rising degradations and requires a cutoff frequency that cannot be derived from this anonymous dataset.
- **Regime inference from RMS amplitude.** Re-labeling active rows as different operating regimes based on magnitude shifts does not generalize across the 29 scenarios.

In [6]:
full_df = clip_rms_spikes(
    full_df,
    vel_threshold=100,
    accel_threshold=10,
)
fit_df = full_df[full_df["split"] == "fit"].copy()

In [7]:
inspector = create_scenario_inspector(full_df, default_scenario=1)
display(inspector)

# 2. Preprocessing Pipeline

## Final Preprocessing Contract

The final preprocessing contract is intentionally narrow:

| Step | Purpose | Kept? |
|---|---|---|
| Spike clipping via `clip_rms_spikes` | Remove sparse gross outliers without redefining the signal shape | Yes |
| `uptime=True` gating | Define which rows count as active machine operation for fitting and scoring | Yes |
| Butterworth or similar frequency-based filtering | Smooth the RMS series using a cutoff / sampling-rate design | No  |
| Regime inference from RMS amplitude | Re-label active rows as ON/OFF based on sensor magnitude | No |

In [8]:
_VEL_RAW = ["vel_rms_x", "vel_rms_y", "vel_rms_z"]
_ACCEL_RAW = ["accel_rms_x", "accel_rms_y", "accel_rms_z"]

full_df_final = full_df.copy()
full_df_final["global_mask_vel"] = ~full_df_final["uptime"].fillna(False).astype(bool)
full_df_final["global_mask_accel"] = ~full_df_final["uptime"].fillna(False).astype(bool)

print(f"global_mask_vel=True:   {full_df_final['global_mask_vel'].sum():,} rows ({full_df_final['global_mask_vel'].mean():.1%})")
print(f"global_mask_accel=True: {full_df_final['global_mask_accel'].sum():,} rows ({full_df_final['global_mask_accel'].mean():.1%})")
print(f"both usable: {(~full_df_final['global_mask_vel'] & ~full_df_final['global_mask_accel']).sum():,} rows")

global_mask_vel=True:   42,355 rows (29.5%)
global_mask_accel=True: 42,355 rows (29.5%)
both usable: 101,319 rows


In [9]:
ui = create_rms_scenario_widget(
    full_df_final,
    default_scenarios=(1, 5, 12),
)
display(ui)

# 3. Healthy Baseline and Residual Calibration

Healthy baselines are fit on the raw RMS channels after spike clipping, using `uptime=True` rows only. Each channel is standardized against its own fit-set mean and standard deviation. The **residual** is the distance above the scoring threshold, in healthy-baseline standard deviations:

```
residual = d_norm - threshold
```

The residual maps to [0, 1] via a sigmoid:

```
score = 1 / (1 + exp(-alpha * (residual - beta)))
```

- **`beta`** shifts the midpoint right — a larger value tolerates larger residuals before the score rises.
- **`alpha`** controls steepness — larger means a sharper jump from near-0 to near-1.

Calibration uses two `(residual, score)` anchor pairs per modality; two anchors and two unknowns (`alpha`, `beta`) give an exact solution. The widget solves it and overlays the fit on the pooled residual histogram, so calibration is visual, not guesswork.

Anchor values change with scenario grouping (`.../norm_model_hyperparams.yaml`); per-group values are derived from this widget once grouping is defined in Section 4.

In [10]:
fit_df = full_df_final[full_df_final["split"] == "fit"].copy()

norm_baselines = fit_norm_baselines(
    df=fit_df,
    scaler="standard",
    vel_cols=_VEL_RAW,
    accel_cols=_ACCEL_RAW,
    vel_mask_col="global_mask_vel",
    accel_mask_col="global_mask_accel",
)

full_scored_df = apply_norm_scores(
    df=full_df_final,
    baselines=norm_baselines,
    vel_cols=_VEL_RAW,
    accel_cols=_ACCEL_RAW,
    vel_mask_col="global_mask_vel",
    accel_mask_col="global_mask_accel",
)

In [11]:
ui, get_sigmoid_params = create_sigmoid_global_residual_widget(
    full_scored_df,
    vel_cols=_VEL_RAW,
    accel_cols=_ACCEL_RAW,
    default_num_bins=100,
    threshold_vel=3.0,
    threshold_accel=3.0,
    alpha_vel=0.74,
    beta_vel=1.86,
    alpha_accel=0.74,
    beta_accel=1.86,
)
display(ui)

In [12]:
sigmoid_params = get_sigmoid_params()
sigmoid_params

{'threshold_vel': 3.0,
 'threshold_accel': 3.0,
 'alpha_vel': 0.74,
 'alpha_accel': 0.74,
 'beta_vel': 1.86,
 'beta_accel': 1.86}

In [13]:
full_scored_df.columns

Index(['scenario_id', 'split', 'label', 'sampled_at', 'uptime', 'vel_rms_x',
       'vel_rms_y', 'vel_rms_z', 'accel_rms_x', 'accel_rms_y', 'accel_rms_z',
       'global_mask_vel', 'global_mask_accel', 'd_vel_rms_x', 'd_vel_rms_y',
       'd_vel_rms_z', 'd_accel_rms_x', 'd_accel_rms_y', 'd_accel_rms_z'],
      dtype='object')

# 4. Manual Grouping From `.pred` Behavior

Scenarios are grouped by their **observed failure archetype** in the residual RMS views — an offline surrogate for missing asset metadata. In production, group membership would come from known attributes like machine type, not retrospective signal inspection.

Each scenario's `pred` residual trajectory was compared against its healthy baseline using the scoring and distribution widgets. Four archetypes emerged (`src/anomaly_detection/model/shared/scenario_groups.py`):

| Group | Archetype | Scenarios | Count |
|---|---|---|---|
| **Group 1** | Large single spike on a flat baseline | 1, 2, 4, 7, 9, 11, 17 | 7 |
| **Group 2** | Sudden sustained trend increase | 3, 6, 10, 15, 16, 18, 20, 21, 22, 24, 26, 27 | 12 |
| **Group 3** | Segments with sharp increases then recovery | 5, 8, 19, 25, 28, 29 | 6 |
| **Group 4** | Many ups and downs with a localized degradation band | 12, 13, 14, 23 | 4 |

A single global `(alpha, beta, threshold, window_top_k)` can't catch both an isolated spike (needs a sharp sigmoid) and a gradual trend (needs a gentler one) without false positives — tested and rejected, hence per-group calibration.

Per-group values are stored in `src/anomaly_detection/model/grouped_residual/hyperparameters/norm_model_hyperparams.yaml` and validated with the group-distribution widget below.

In [14]:
full_df = add_scenario_group_labels(full_df)
full_df_final = add_scenario_group_labels(full_df_final)
full_scored_df = add_scenario_group_labels(full_scored_df)

full_scored_df[["scenario_id", "scenario_group", "scenario_group_label"]].drop_duplicates().sort_values("scenario_id").reset_index(drop=True)

,scenario_id,scenario_group,scenario_group_label
0,1,group_1,Group 1 - large single spike
1,2,group_1,Group 1 - large single spike
2,3,group_2,Group 2 - sudden trend increase
3,4,group_1,Group 1 - large single spike
4,5,group_3,Group 3 - segments with sudden increases and b...
5,6,group_2,Group 2 - sudden trend increase
6,7,group_1,Group 1 - large single spike
7,8,group_3,Group 3 - segments with sudden increases and b...
8,9,group_1,Group 1 - large single spike
9,10,group_2,Group 2 - sudden trend increase


In [15]:
ui = create_group_distribution_widget(
    full_scored_df,
    vel_cols=_VEL_RAW,
    accel_cols=_ACCEL_RAW,
    threshold_vel=sigmoid_params["threshold_vel"],
    threshold_accel=sigmoid_params["threshold_accel"],
)
display(ui)


### Handoff to Model Debugging

The assumptions above — spike clipping only, `uptime=True` gating, per-group calibration — carry forward unchanged into `1.02-acp-model-debugging.ipynb`, which fits all 29 scenarios and validates alert engine behavior against labeled incident windows.